In [0]:
#TxUTuvrZZJ9LMKhOK6GI81YaniXXrBw2TfnA08X9fgQ1EeQ4sS20rGZmaRjnWqEswZe4l8LMmHsa+AStSuqpyQ==

storage_account = "tlcyifanadelaide2026"

spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  "TxUTuvrZZJ9LMKhOK6GI81YaniXXrBw2TfnA08X9fgQ1EeQ4sS20rGZmaRjnWqEswZe4l8LMmHsa+AStSuqpyQ=="
)

In [0]:
raw_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/yellowtaxi/"
silver_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/silver/tlc_yellow_trips/"
rejects_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/silver/tlc_yellow_trips_rejected/"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

raw_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/yellowtaxi/"
files = [f.path for f in dbutils.fs.ls(raw_path) if f.path.endswith(".parquet")]
len(files), files[:3]

(119,
 ['abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/yellowtaxi/yellow_tripdata_2016-01.parquet',
  'abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/yellowtaxi/yellow_tripdata_2016-02.parquet',
  'abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/yellowtaxi/yellow_tripdata_2016-03.parquet'])

In [0]:
def normalize(df):
    # make sure columns exist
    expected_cols = [
        "VendorID","tpep_pickup_datetime","tpep_dropoff_datetime","passenger_count","trip_distance",
        "RatecodeID","store_and_fwd_flag","PULocationID","DOLocationID","payment_type",
        "fare_amount","extra","mta_tax","tip_amount","tolls_amount","improvement_surcharge",
        "congestion_surcharge","airport_fee","total_amount"
    ]
    for c in expected_cols:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None))

    return (
        df
        .withColumn("VendorID", F.col("VendorID").cast("int"))
        .withColumn("pickup_ts", F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
        # IMPORTANT: cast passenger_count via double -> int to handle both bigint/double sources
        .withColumn("passenger_count", F.col("passenger_count").cast("double").cast("int"))
        .withColumn("trip_distance", F.col("trip_distance").cast("double"))
        .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
        .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast("string"))
        .withColumn("PULocationID", F.col("PULocationID").cast("int"))
        .withColumn("DOLocationID", F.col("DOLocationID").cast("int"))
        .withColumn("payment_type", F.col("payment_type").cast("int"))
        .withColumn("pickup_date", F.to_date("pickup_ts"))
        .withColumn("pickup_year", F.year("pickup_ts").cast("int"))
        .withColumn("pickup_month", F.month("pickup_ts").cast("int"))
        .withColumn("trip_duration_sec",
                    (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")).cast("long"))
    )

money_cols = [
    "fare_amount","extra","mta_tax","tip_amount","tolls_amount",
    "improvement_surcharge","congestion_surcharge","airport_fee","total_amount"
]

money_dec = T.DecimalType(10, 2)

def cast_money_double(df):
    for c in money_cols:
        df = df.withColumn(c, F.col(c).cast("double"))
    return df

def filter_money_nonneg(df):
    cond = F.lit(True)
    for c in money_cols:
        cond = cond & F.col(c).isNotNull() & (F.col(c) >= 0)
    return df.filter(cond)

def cast_money_decimal(df):
    for c in money_cols:
        df = df.withColumn(c, F.col(c).cast(money_dec))
    return df


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

silver_path  = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/silver/tlc_yellow_trips_v2/"
rejects_path = "abfss://rawdata@tlcyifanadelaide2026.dfs.core.windows.net/silver/tlc_yellow_trips_rejected_v2/"

min_ts = F.to_timestamp(F.lit("2016-01-01"))
max_ts = F.date_add(F.current_timestamp(), 1)

# clean rebuild (avoid partial overwrite)
dbutils.fs.rm(silver_path, True)
dbutils.fs.rm(rejects_path, True)

failed = []
processed = 0

required_money = ["fare_amount", "total_amount"]
optional_money = [
    "extra","mta_tax","tip_amount","tolls_amount","improvement_surcharge",
    "congestion_surcharge","airport_fee"
]

for p in sorted(files):
    try:
        df_one = spark.read.parquet(p)

        # optional drop
        if "cbd_congestion_fee" in df_one.columns:
            df_one = df_one.drop("cbd_congestion_fee")

        # normalize (NO decimal casts inside normalize if possible)
        df_norm = normalize(df_one)

        # cast money to double for safe checks
        df_norm = cast_money_double(df_norm)

        # build conditions on THIS df
        bad_ts = (
            F.col("pickup_ts").isNull() |
            F.col("dropoff_ts").isNull() |
            (F.col("pickup_ts") < min_ts) |
            (F.col("pickup_ts") > max_ts) |
            (F.col("dropoff_ts") < F.col("pickup_ts"))
        )

        money_ok = F.lit(True)
        for c in required_money:
            money_ok = money_ok & F.col(c).isNotNull() & (F.col(c) >= 0.0)
        for c in optional_money:
            money_ok = money_ok & (F.col(c).isNull() | (F.col(c) >= 0.0))

        bad = bad_ts | (~money_ok)

        # split BEFORE casting to decimal
        acc = df_norm.filter(~bad)
        rej = (df_norm.filter(bad)
               .withColumn("reject_reason",
                           F.when(bad_ts, F.lit("bad_timestamp"))
                            .otherwise(F.lit("bad_money")))
               .withColumn("source_file", F.lit(p))
              )

        # now cast accepted money to decimal (safe after filter)
        acc = cast_money_decimal(acc)

        (acc.write
            .format("delta")
            .mode("append")
            .partitionBy("pickup_year", "pickup_month")
            .save(silver_path)
        )

        (rej.write
            .format("delta")
            .mode("append")
            .save(rejects_path)
        )

        processed += 1

    except Exception as e:
        failed.append((p, str(e)[:300]))

processed, len(failed), failed[:5]

(119, 0, [])